In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1ACeyFLJkmGQexnmBdSZGBpqc9OTXj_sj", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


# Notebook 1: Ring-AllReduce Simulation

**Vizuara AI Pods | GPU Programming Course | Pod 4: Ring-AllReduce, Batch Size, and Profiling**

---

In this notebook, we will build a **ring-allreduce algorithm from scratch** in pure Python. By simulating the algorithm step by step, you will gain an intuitive understanding of:

1. Why naive allreduce creates a single-GPU bottleneck
2. How ring-allreduce eliminates that bottleneck with its scatter-reduce and allgather phases
3. Why ring-allreduce is bandwidth-optimal
4. How PyTorch's `DistributedDataParallel` uses this under the hood

**Prerequisites:** Basic Python and NumPy knowledge.

**Estimated time:** 40-50 minutes

**Runtime:** CPU is fine for this notebook -- we are simulating the algorithm, not running it on real GPUs.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Setup

In [ ]:
!pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from copy import deepcopy
from typing import List, Optional

np.random.seed(42)
print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Part1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_part1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 1: The Communication Problem

In data parallelism, every GPU holds a copy of the model and computes gradients on its own mini-batch. After the backward pass, all GPUs must end up with the **same averaged gradient** so they can take identical optimizer steps.

This operation is called **allreduce**: take a vector that exists on every GPU, sum (or average) it, and place the result on every GPU.

Let's set up a simulation. We will represent each GPU as holding a gradient vector, and track all communication.

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Gpu Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_simulated_gpu_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimulatedGPU:
    """Represents one GPU in our distributed training simulation."""

    def __init__(self, gpu_id: int, gradient: np.ndarray):
        self.gpu_id = gpu_id
        self.data = gradient.copy()
        self.bytes_sent = 0
        self.bytes_received = 0

    def send(self, chunk: np.ndarray) -> np.ndarray:
        """Send data to another GPU. Returns the data and tracks bytes."""
        self.bytes_sent += chunk.nbytes
        return chunk.copy()

    def receive(self, chunk: np.ndarray):
        """Receive data from another GPU. Tracks bytes."""
        self.bytes_received += chunk.nbytes
        return chunk.copy()

    def __repr__(self):
        return f"GPU({self.gpu_id}, data={self.data}, sent={self.bytes_sent}B, recv={self.bytes_received}B)"


def create_gpus(num_gpus: int, gradient_size: int) -> List[SimulatedGPU]:
    """Create N simulated GPUs, each with a random gradient vector."""
    gpus = []
    for i in range(num_gpus):
        # Each GPU has a different gradient (from its own mini-batch)
        gradient = np.random.randn(gradient_size).astype(np.float32)
        gpus.append(SimulatedGPU(i, gradient))
    return gpus


# Create 4 GPUs with gradient vectors of size 8 (small for visualization)
NUM_GPUS = 4
GRAD_SIZE = 8
gpus = create_gpus(NUM_GPUS, GRAD_SIZE)

# Compute the expected result: average of all gradients
expected_result = np.mean([gpu.data for gpu in gpus], axis=0)

print(f"Number of GPUs: {NUM_GPUS}")
print(f"Gradient size: {GRAD_SIZE} elements ({GRAD_SIZE * 4} bytes)")
print(f"\nInitial gradients:")
for gpu in gpus:
    print(f"  GPU {gpu.gpu_id}: {gpu.data[:4]}... (showing first 4 elements)")
print(f"\nExpected average: {expected_result[:4]}...")

In [ ]:
#@title 🎧 Listen: Naive Allreduce Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_naive_allreduce_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 2: Naive AllReduce (The Bad Approach)

The simplest approach: designate GPU 0 as the "master." Every other GPU sends its gradient to GPU 0, which averages them and broadcasts the result back.

Let's implement it and measure how much data the master has to handle.

In [ ]:
#@title 🎧 Code Walkthrough: Naive Allreduce Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_naive_allreduce_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def naive_allreduce(gpus: List[SimulatedGPU]) -> List[SimulatedGPU]:
    """Naive allreduce: gather to master (GPU 0), average, broadcast back."""
    gpus = deepcopy(gpus)  # Don't modify originals
    N = len(gpus)
    master = gpus[0]

    # Phase 1: Gather -- all GPUs send gradients to master
    accumulated = master.data.copy()
    for i in range(1, N):
        sent_data = gpus[i].send(gpus[i].data)  # GPU i sends its gradient
        received = master.receive(sent_data)      # Master receives it
        accumulated += received

    # Average
    averaged = accumulated / N
    master.data = averaged.copy()

    # Phase 2: Broadcast -- master sends the averaged gradient to all others
    for i in range(1, N):
        sent_data = master.send(averaged)      # Master sends to GPU i
        gpus[i].data = gpus[i].receive(sent_data)  # GPU i receives

    return gpus


# Run naive allreduce
naive_result = naive_allreduce(gpus)


In [ ]:
#@title 🎧 Listen: Naive Allreduce Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_naive_allreduce_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("Naive AllReduce Results:")
print("=" * 60)

# Check correctness
for gpu in naive_result:
    error = np.max(np.abs(gpu.data - expected_result))
    print(f"  GPU {gpu.gpu_id}: max error = {error:.2e}  "
          f"sent={gpu.bytes_sent}B  recv={gpu.bytes_received}B")

master = naive_result[0]
total_master_traffic = master.bytes_sent + master.bytes_received
grad_bytes = GRAD_SIZE * 4  # float32

print(f"\nMaster (GPU 0) total traffic: {total_master_traffic} bytes")
print(f"That is {total_master_traffic / grad_bytes:.1f}x the gradient size")
print(f"Formula: 2 * (N-1) * D = 2 * {NUM_GPUS-1} * {grad_bytes} = {2 * (NUM_GPUS-1) * grad_bytes}")

In [ ]:
#@title 🎧 Listen: Naive Scaling Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_naive_scaling_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Scaling Analysis: Why Naive AllReduce Breaks

Let's see what happens to the master's traffic as we scale to more GPUs.

In [ ]:
gpu_counts = [2, 4, 8, 16, 32, 64]
grad_size = 1_000_000  # 1M parameters (realistic small model)
grad_bytes = grad_size * 4  # float32

naive_master_traffic = []
naive_max_per_gpu = []

print(f"Gradient size: {grad_size:,} params = {grad_bytes / 1e6:.1f} MB")
print("=" * 60)

for N in gpu_counts:
    # Master receives from N-1 GPUs, sends to N-1 GPUs
    master_traffic = 2 * (N - 1) * grad_bytes
    naive_master_traffic.append(master_traffic)
    naive_max_per_gpu.append(master_traffic)

    print(f"  N={N:>3} GPUs: Master traffic = {master_traffic / 1e6:8.1f} MB  "
          f"({master_traffic / grad_bytes:.0f}x gradient size)")

print(f"\nWith 64 GPUs, the master handles {naive_master_traffic[-1] / 1e9:.1f} GB of traffic!")
print("Every other GPU sits idle while the master struggles.")

In [ ]:
#@title 🎧 Listen: Ring Allreduce Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_ring_allreduce_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 3: Ring-AllReduce (The Right Approach)

Ring-allreduce arranges all GPUs in a logical ring. Each GPU only talks to its two neighbors. The algorithm has two phases:

1. **Scatter-Reduce:** Each GPU accumulates the sum for one chunk. After N-1 steps, each GPU holds one fully reduced chunk.
2. **Allgather:** The fully reduced chunks are propagated around the ring until every GPU has all of them.

Let's implement it step by step.

In [ ]:
#@title 🎧 Code Walkthrough: Ring Allreduce Code Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_ring_allreduce_code_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def ring_allreduce(gpus: List[SimulatedGPU], verbose: bool = True) -> List[SimulatedGPU]:
    """
    Ring-AllReduce: bandwidth-optimal allreduce.

    Phase 1 (Scatter-Reduce): Each GPU accumulates one chunk.
    Phase 2 (Allgather): Fully reduced chunks propagate to all GPUs.
    """
    gpus = deepcopy(gpus)
    N = len(gpus)
    D = len(gpus[0].data)
    chunk_size = D // N

    # Split each GPU's data into N chunks
    chunks = []
    for gpu in gpus:
        gpu_chunks = [gpu.data[j * chunk_size:(j + 1) * chunk_size].copy() for j in range(N)]
        chunks.append(gpu_chunks)

    if verbose:
        print(f"Ring-AllReduce: {N} GPUs, {D} elements, {chunk_size} elements/chunk")
        print("=" * 70)

    # =============================================
    # PHASE 1: Scatter-Reduce (N-1 steps)
    # =============================================
    if verbose:


In [ ]:
#@title 🎧 Code Walkthrough: Ring Allreduce Phase1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_ring_allreduce_phase1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
        print("\n--- Phase 1: Scatter-Reduce ---")

    for step in range(N - 1):
        if verbose:
            print(f"\n  Step {step + 1}:")

        # Each GPU sends one chunk to its next neighbor
        # and receives one chunk from its previous neighbor
        new_chunks = [row[:] for row in chunks]  # shallow copy of chunk lists

        for i in range(N):
            # GPU i sends chunk at index (i - step) % N to GPU (i+1) % N
            send_idx = (i - step) % N
            recv_from = (i - 1) % N
            send_to = (i + 1) % N

            # The chunk GPU i sends
            sent_data = gpus[i].send(chunks[i][send_idx])

            # GPU (i+1) receives and accumulates
            received = gpus[send_to].receive(sent_data)
            new_chunks[send_to][send_idx] = chunks[send_to][send_idx] + received

            if verbose and i == 0:
                print(f"    GPU {i} sends chunk {send_idx} -> GPU {send_to}")

        chunks = new_chunks

    if verbose:
        print("\n  After Scatter-Reduce:")
        for i in range(N):
            reduced_chunk_idx = (i + 1) % N
            print(f"    GPU {i} owns fully reduced chunk {reduced_chunk_idx}")

    # =============================================
    # PHASE 2: Allgather (N-1 steps)
    # =============================================
    if verbose:


In [ ]:
#@title 🎧 Code Walkthrough: Ring Allreduce Phase2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_ring_allreduce_phase2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
        print("\n--- Phase 2: Allgather ---")

    for step in range(N - 1):
        if verbose:
            print(f"\n  Step {step + 1}:")

        new_chunks = [row[:] for row in chunks]

        for i in range(N):
            # GPU i sends the chunk it most recently received (or its own reduced chunk)
            send_idx = (i + 1 - step) % N
            send_to = (i + 1) % N

            sent_data = gpus[i].send(chunks[i][send_idx])
            received = gpus[send_to].receive(sent_data)

            # In allgather, we REPLACE (not accumulate)
            new_chunks[send_to][send_idx] = received

            if verbose and i == 0:
                print(f"    GPU {i} sends chunk {send_idx} -> GPU {send_to}")

        chunks = new_chunks

    # Reconstruct the full gradient on each GPU and divide by N
    for i in range(N):
        gpus[i].data = np.concatenate(chunks[i]) / N

    return gpus


# Run ring-allreduce
ring_result = ring_allreduce(gpus, verbose=True)

In [ ]:
#@title 🎧 Listen: Ring Allreduce Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_ring_allreduce_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verify correctness
print("\nRing-AllReduce Results:")
print("=" * 60)

for gpu in ring_result:
    error = np.max(np.abs(gpu.data - expected_result))
    print(f"  GPU {gpu.gpu_id}: max error = {error:.2e}  "
          f"sent={gpu.bytes_sent}B  recv={gpu.bytes_received}B")

# Compare traffic: all GPUs should have the same traffic
max_traffic_ring = max(g.bytes_sent + g.bytes_received for g in ring_result)
grad_bytes = GRAD_SIZE * 4

print(f"\nMax traffic per GPU: {max_traffic_ring} bytes")
print(f"Formula: 2 * (N-1) * D/N = 2 * {NUM_GPUS-1} * {grad_bytes//NUM_GPUS} = {2 * (NUM_GPUS-1) * grad_bytes // NUM_GPUS}")
print(f"\nCompare with naive master traffic: {2 * (NUM_GPUS-1) * grad_bytes} bytes")
print(f"Ring is {2*(NUM_GPUS-1)*grad_bytes / max_traffic_ring:.1f}x less traffic at the bottleneck!")

In [ ]:
#@title 🎧 Listen: Visualization Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_visualization_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 4: Visualizing the Algorithm

Let's create a visual trace of the ring-allreduce algorithm to see exactly what happens at each step.

In [ ]:
#@title 🎧 What to Look For: Visualization Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_visualization_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def visualize_ring_allreduce(num_gpus=4):
    """
    Visualize the ring-allreduce algorithm step by step.
    Shows which chunks each GPU holds and their reduction state.
    """
    N = num_gpus
    colors = plt.cm.Set2(np.linspace(0, 1, N))

    # Track the state: for each GPU, for each chunk, track how many GPUs
    # have contributed to it
    # state[gpu][chunk] = number of GPUs summed into that chunk
    states = []

    # Initial state: each GPU has its own contribution (count = 1)
    initial = [[1 for _ in range(N)] for _ in range(N)]
    states.append(deepcopy(initial))

    # Scatter-reduce: N-1 steps
    current = deepcopy(initial)
    for step in range(N - 1):
        new_state = deepcopy(current)
        for i in range(N):
            send_idx = (i - step) % N
            send_to = (i + 1) % N
            new_state[send_to][send_idx] = current[send_to][send_idx] + current[i][send_idx]
        current = new_state
        states.append(deepcopy(current))

    # Allgather: N-1 steps
    for step in range(N - 1):
        new_state = deepcopy(current)
        for i in range(N):
            send_idx = (i + 1 - step) % N
            send_to = (i + 1) % N
            new_state[send_to][send_idx] = current[i][send_idx]
        current = new_state
        states.append(deepcopy(current))

    # Plot
    total_steps = len(states)
    fig, axes = plt.subplots(1, total_steps, figsize=(3 * total_steps, 4))

    step_labels = ["Initial"]
    for s in range(1, N):
        step_labels.append(f"SR Step {s}")
    for s in range(1, N):
        step_labels.append(f"AG Step {s}")

    for t, (state, ax) in enumerate(zip(states, axes)):
        ax.set_title(step_labels[t], fontsize=10, fontweight='bold')
        ax.set_xlim(-0.5, N - 0.5)
        ax.set_ylim(-0.5, N - 0.5)
        ax.set_xticks(range(N))
        ax.set_xticklabels([f"Chunk {j}" for j in range(N)], fontsize=7, rotation=45)
        ax.set_yticks(range(N))
        ax.set_yticklabels([f"GPU {i}" for i in range(N)], fontsize=8)

        for i in range(N):
            for j in range(N):
                count = state[i][j]
                # Color intensity based on how many GPUs contributed
                intensity = count / N
                color = plt.cm.YlOrRd(intensity * 0.8 + 0.1)
                rect = plt.Rectangle((j - 0.4, i - 0.4), 0.8, 0.8,
                                    facecolor=color, edgecolor='black', linewidth=0.5)
                ax.add_patch(rect)
                ax.text(j, i, str(count), ha='center', va='center',
                       fontsize=12, fontweight='bold')

        ax.invert_yaxis()
        ax.set_aspect('equal')

    plt.suptitle('Ring-AllReduce: Number of GPUs Contributing to Each Chunk\n'
                 f'(N={N} GPUs, fully reduced = {N})',
                 fontsize=13, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()


In [ ]:
#@title 🎧 Listen: Visualization Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_visualization_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    print("Numbers show how many GPUs have been summed into each chunk.")
    print(f"When a cell shows '{N}', that chunk is fully reduced (sum of all {N} GPUs).")
    print("SR = Scatter-Reduce, AG = Allgather")

visualize_ring_allreduce(4)

In [ ]:
#@title 🎧 Listen: Bandwidth Analysis Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_bandwidth_analysis_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 5: Bandwidth Analysis

The key theoretical result: ring-allreduce's per-GPU traffic approaches `2D` (twice the gradient size) as the number of GPUs grows, while naive allreduce's bottleneck traffic grows as `2(N-1)*D`.

Let's compute and visualize this scaling.

In [ ]:
#@title 🎧 Code Walkthrough: Bandwidth Analysis Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_bandwidth_analysis_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
gpu_counts = [2, 4, 8, 16, 32, 64, 128, 256]
D = 100_000_000  # 100M parameters (e.g., a small language model)
bytes_per_param = 4  # float32
D_bytes = D * bytes_per_param

naive_bottleneck = []  # Traffic at the master GPU
ring_per_gpu = []      # Traffic per GPU in ring

print(f"Model size: {D:,} parameters = {D_bytes / 1e9:.1f} GB")
print("=" * 75)
print(f"{'N GPUs':>8}  {'Naive Bottleneck':>18}  {'Ring per GPU':>18}  {'Ratio':>8}")
print("-" * 75)

for N in gpu_counts:
    # Naive: master handles 2*(N-1)*D bytes
    naive = 2 * (N - 1) * D_bytes
    naive_bottleneck.append(naive)

    # Ring: each GPU handles 2*(N-1)/N * D bytes
    ring = 2 * (N - 1) / N * D_bytes
    ring_per_gpu.append(ring)

    ratio = naive / ring if ring > 0 else float('inf')
    print(f"{N:>8}  {naive/1e9:>15.1f} GB  {ring/1e9:>15.1f} GB  {ratio:>7.1f}x")

print(f"\nAs N -> infinity, ring per-GPU traffic -> 2D = {2 * D_bytes / 1e9:.1f} GB")
print(f"Naive bottleneck at N=256: {naive_bottleneck[-1] / 1e12:.1f} TB (!)")

In [ ]:
#@title 🎧 What to Look For: Bandwidth Analysis Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_bandwidth_analysis_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the scaling difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Absolute traffic
ax = axes[0]
ax.plot(gpu_counts, [x / 1e9 for x in naive_bottleneck], 'o--', color='#E53935',
        linewidth=2, markersize=8, label='Naive (master bottleneck)')
ax.plot(gpu_counts, [x / 1e9 for x in ring_per_gpu], 's-', color='#1E88E5',
        linewidth=2, markersize=8, label='Ring (per GPU)')
ax.axhline(y=2 * D_bytes / 1e9, color='#1E88E5', linestyle=':', alpha=0.5,
           label=f'Ring limit: 2D = {2*D_bytes/1e9:.0f} GB')
ax.set_xlabel('Number of GPUs', fontsize=12)
ax.set_ylabel('Max Traffic per GPU (GB)', fontsize=12)
ax.set_title('Communication Traffic Scaling', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Plot 2: Normalized traffic (relative to gradient size)
ax = axes[1]
ax.plot(gpu_counts, [x / D_bytes for x in naive_bottleneck], 'o--', color='#E53935',
        linewidth=2, markersize=8, label='Naive (master): O(N*D)')
ax.plot(gpu_counts, [x / D_bytes for x in ring_per_gpu], 's-', color='#1E88E5',
        linewidth=2, markersize=8, label='Ring (per GPU): O(D)')
ax.axhline(y=2, color='#1E88E5', linestyle=':', alpha=0.5, label='Ring limit: 2')
ax.set_xlabel('Number of GPUs', fontsize=12)
ax.set_ylabel('Traffic / Gradient Size', fontsize=12)
ax.set_title('Normalized Communication Cost', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Ring-allreduce's per-GPU cost is nearly constant.")
print("Adding more GPUs does NOT significantly increase communication time.")

In [ ]:
#@title 🎧 Listen: Comm Time Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_comm_time_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 6: Simulating Communication Time

Let's simulate the actual communication time assuming a specific network bandwidth. This makes the advantage concrete.

In [ ]:
#@title 🎧 Code Walkthrough: Comm Time Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_comm_time_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def estimate_allreduce_time(num_gpus, gradient_bytes, bandwidth_gbps, latency_us=5):
    """
    Estimate allreduce time for both naive and ring approaches.

    Args:
        num_gpus: Number of GPUs
        gradient_bytes: Total gradient size in bytes
        bandwidth_gbps: Network bandwidth in GB/s (per link)
        latency_us: Per-message latency in microseconds

    Returns:
        dict with timing estimates
    """
    N = num_gpus
    D = gradient_bytes
    B = bandwidth_gbps * 1e9  # Convert to bytes/sec
    L = latency_us * 1e-6     # Convert to seconds

    # Naive: master must receive from N-1, then send to N-1
    # Sequential at the master
    naive_time = 2 * (N - 1) * (D / B + L)

    # Ring: 2*(N-1) steps, each transferring D/N bytes
    # All steps are pipelined (all GPUs send/recv simultaneously)
    ring_time = 2 * (N - 1) * (D / (N * B) + L)

    return {
        'naive_ms': naive_time * 1000,
        'ring_ms': ring_time * 1000,
        'speedup': naive_time / ring_time if ring_time > 0 else float('inf')
    }


# Realistic scenario: 100M parameter model, 100 Gbps NVLink
grad_bytes = 100_000_000 * 4  # 400 MB gradient (float32)
bandwidth = 25  # 25 GB/s (approximate per-link NVLink bandwidth)

print(f"Model: 100M params, Gradient: {grad_bytes/1e6:.0f} MB, Bandwidth: {bandwidth} GB/s")
print("=" * 65)
print(f"{'N GPUs':>8}  {'Naive':>12}  {'Ring':>12}  {'Speedup':>10}")
print("-" * 65)

gpu_counts_sim = [2, 4, 8, 16, 32, 64, 128]
naive_times = []
ring_times = []

for N in gpu_counts_sim:
    result = estimate_allreduce_time(N, grad_bytes, bandwidth)
    naive_times.append(result['naive_ms'])
    ring_times.append(result['ring_ms'])
    print(f"{N:>8}  {result['naive_ms']:>10.2f}ms  {result['ring_ms']:>10.2f}ms  {result['speedup']:>9.1f}x")

In [ ]:
#@title 🎧 What to Look For: Comm Time Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_comm_time_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize communication time
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(gpu_counts_sim, naive_times, 'o--', color='#E53935', linewidth=2,
        markersize=8, label='Naive AllReduce')
ax.plot(gpu_counts_sim, ring_times, 's-', color='#1E88E5', linewidth=2,
        markersize=8, label='Ring-AllReduce')

ax.set_xlabel('Number of GPUs', fontsize=13)
ax.set_ylabel('AllReduce Time (ms)', fontsize=13)
ax.set_title('AllReduce Communication Time\n'
             f'(100M params, {bandwidth} GB/s per link)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

# Annotate the key insight
ax.annotate(f'Ring stays near {ring_times[-1]:.0f}ms\neven at 128 GPUs',
            xy=(128, ring_times[-1]), xytext=(32, ring_times[-1] + 200),
            fontsize=11, ha='center',
            arrowprops=dict(arrowstyle='->', color='#1E88E5'),
            color='#1E88E5', fontweight='bold')

ax.annotate(f'Naive: {naive_times[-1]:.0f}ms\nat 128 GPUs!',
            xy=(128, naive_times[-1]), xytext=(32, naive_times[-1] - 800),
            fontsize=11, ha='center',
            arrowprops=dict(arrowstyle='->', color='#E53935'),
            color='#E53935', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Bucketing Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_bucketing_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 7: Gradient Bucketing

In practice, PyTorch does not do one giant allreduce for the entire gradient. It groups parameters into **buckets** and launches an allreduce for each bucket as soon as all gradients in that bucket are ready.

This allows **overlapping communication with computation** during the backward pass.

Let's simulate this to understand the trade-off.

In [ ]:
#@title 🎧 Code Walkthrough: Bucketing Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_bucketing_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def simulate_bucketed_training_step(
    num_layers: int,
    params_per_layer: int,
    backward_time_per_layer_ms: float,
    bucket_size_params: int,
    num_gpus: int,
    bandwidth_gbps: float
):
    """
    Simulate a training step with gradient bucketing and communication overlap.

    Returns timing information for the backward + allreduce phase.
    """
    total_params = num_layers * params_per_layer
    params_per_bucket = bucket_size_params
    num_buckets = (total_params + params_per_bucket - 1) // params_per_bucket

    N = num_gpus
    B = bandwidth_gbps * 1e9  # bytes/sec

    # Time for ring-allreduce of one bucket
    bucket_bytes = params_per_bucket * 4  # float32
    allreduce_time_per_bucket = 2 * (N - 1) / N * bucket_bytes / B * 1000  # ms

    # Without overlap: backward + allreduce sequentially
    total_backward = num_layers * backward_time_per_layer_ms
    total_allreduce = num_buckets * allreduce_time_per_bucket
    time_no_overlap = total_backward + total_allreduce

    # With overlap: allreduce starts as soon as each bucket's gradients are ready
    # Backward computes gradients from last layer to first (reverse order)
    # Buckets are also in reverse order (matching backward computation)
    # Communication can overlap with remaining backward computation

    # The exposed (non-overlapped) communication time is the allreduce time
    # for the LAST bucket (which finishes after backward is done)
    # minus any backward computation that happens concurrently

    # Simplified model: only the last bucket's allreduce is fully exposed
    if total_allreduce <= total_backward:
        # Communication fully hidden behind backward
        time_with_overlap = total_backward + allreduce_time_per_bucket
    else:
        # Some communication exposed
        time_with_overlap = total_backward + (total_allreduce - total_backward) + allreduce_time_per_bucket

    return {
        'num_buckets': num_buckets,
        'backward_ms': total_backward,
        'allreduce_ms': total_allreduce,
        'time_no_overlap_ms': time_no_overlap,
        'time_with_overlap_ms': time_with_overlap,
        'overlap_savings_pct': (1 - time_with_overlap / time_no_overlap) * 100,
        'allreduce_per_bucket_ms': allreduce_time_per_bucket
    }


# Simulate different bucket sizes
NUM_LAYERS = 24
PARAMS_PER_LAYER = 10_000_000  # 10M params per layer
BACKWARD_TIME = 5.0  # ms per layer
NUM_GPUS_SIM = 8
BANDWIDTH = 25.0  # GB/s

bucket_sizes = [1_000_000, 5_000_000, 10_000_000, 25_000_000, 50_000_000, 100_000_000, 240_000_000]

print(f"Model: {NUM_LAYERS} layers x {PARAMS_PER_LAYER/1e6:.0f}M params = {NUM_LAYERS*PARAMS_PER_LAYER/1e6:.0f}M total")
print(f"GPUs: {NUM_GPUS_SIM}, Bandwidth: {BANDWIDTH} GB/s")
print("=" * 85)
print(f"{'Bucket Size':>14}  {'#Buckets':>8}  {'No Overlap':>12}  {'With Overlap':>14}  {'Savings':>8}")
print("-" * 85)

for bs in bucket_sizes:
    r = simulate_bucketed_training_step(NUM_LAYERS, PARAMS_PER_LAYER, BACKWARD_TIME, bs, NUM_GPUS_SIM, BANDWIDTH)
    print(f"{bs/1e6:>11.0f}M  {r['num_buckets']:>8}  {r['time_no_overlap_ms']:>10.1f}ms  "
          f"{r['time_with_overlap_ms']:>12.1f}ms  {r['overlap_savings_pct']:>7.1f}%")

## Exercises


In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_29_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Implement Scatter-Reduce Only

Implement **just the scatter-reduce phase** of ring-allreduce. After scatter-reduce, each GPU should hold exactly one fully reduced chunk (the sum of that chunk from all GPUs). Verify your implementation by checking that GPU `k` holds the correct sum for chunk `(k+1) % N`.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_30_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def scatter_reduce_only(gradients: List[np.ndarray]) -> List[np.ndarray]:
    """
    Implement only the scatter-reduce phase of ring-allreduce.

    Args:
        gradients: List of gradient vectors, one per GPU.

    Returns:
        List of chunk lists. After scatter-reduce, GPU k should have
        the fully reduced version of chunk (k+1) % N.
    """
    N = len(gradients)
    D = len(gradients[0])
    chunk_size = D // N

    # Split each gradient into N chunks
    chunks = []
    for g in gradients:
        gpu_chunks = [g[j * chunk_size:(j + 1) * chunk_size].copy() for j in range(N)]
        chunks.append(gpu_chunks)

    # TODO: Implement N-1 steps of scatter-reduce
    # In each step:
    #   - GPU i sends chunk (i - step) % N to GPU (i + 1) % N
    #   - The receiving GPU ADDS the received chunk to its own chunk at that index
    #
    # for step in range(N - 1):
    #     ...

    pass  # TODO: Replace with your implementation

    return chunks


# Test your implementation
N_test = 4
D_test = 16
test_grads = [np.random.randn(D_test).astype(np.float32) for _ in range(N_test)]
expected_sum = np.sum(test_grads, axis=0)

result_chunks = scatter_reduce_only(test_grads)

if result_chunks is not None:
    chunk_size = D_test // N_test
    print("Verification:")
    for i in range(N_test):
        reduced_idx = (i + 1) % N_test
        expected_chunk = expected_sum[reduced_idx * chunk_size:(reduced_idx + 1) * chunk_size]
        actual_chunk = result_chunks[i][reduced_idx]
        error = np.max(np.abs(actual_chunk - expected_chunk))
        status = "PASS" if error < 1e-5 else "FAIL"
        print(f"  GPU {i}, chunk {reduced_idx}: error = {error:.2e} [{status}]")
else:


In [ ]:
#@title 🎧 Before You Start: Todo1 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_31_todo1_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    print("TODO: Implement scatter_reduce_only()")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_32_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Analyze Latency-Dominated Scenarios

Ring-allreduce requires `2*(N-1)` communication steps, each with a startup latency. For very small messages, this latency can dominate the actual data transfer time.

Plot the total allreduce time (including latency) for ring-allreduce as a function of gradient size, for different numbers of GPUs. At what gradient size does the bandwidth term start to dominate the latency term?

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_33_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Latency vs Bandwidth analysis
#
# For ring-allreduce, total time = 2*(N-1) * (D/(N*B) + L)
#   where D = gradient bytes, B = bandwidth (bytes/s), L = per-step latency (s)
#
# The crossover point where bandwidth term equals latency term:
#   D/(N*B) = L  -->  D = N * B * L
#
# Plot total time vs gradient size for N = 4, 8, 16, 32, 64
# with bandwidth = 25 GB/s and latency = 5 microseconds

bandwidth_bps = 25e9  # 25 GB/s
latency_s = 5e-6      # 5 microseconds

grad_sizes = np.logspace(3, 9, 50)  # 1KB to 1GB
gpu_counts_ex = [4, 8, 16, 32, 64]

# TODO: For each GPU count, compute the allreduce time for each gradient size
# TODO: Plot the results (use log-log scale)
# TODO: Mark the crossover point where bandwidth term = latency term

# fig, ax = plt.subplots(figsize=(10, 6))
# for N in gpu_counts_ex:
#     times = [...]  # TODO: compute times
#     ax.loglog(grad_sizes, times, label=f'N={N}')
#     crossover = N * bandwidth_bps * latency_s
#     ax.axvline(x=crossover, linestyle=':', alpha=0.3)
# ...


In [ ]:
#@title 🎧 Before You Start: Todo2 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_34_todo2_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("TODO: Implement the latency vs bandwidth analysis")
print("Hint: Use the formula T = 2*(N-1) * (D/(N*B) + L)")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_35_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 3: Tree-AllReduce Comparison

Another popular allreduce algorithm is **tree-allreduce** (also called recursive halving-doubling). In tree-allreduce, GPUs communicate in a binary tree pattern:

- **Reduce phase:** Pairs of GPUs exchange half their data, reduce it, and repeat with halved groups for `log2(N)` steps.
- **Broadcast phase:** The result is distributed back through the tree.

Total per-GPU traffic: `2 * log2(N) * D/2`

Implement a function that computes tree-allreduce's communication cost and compare it with ring-allreduce across different GPU counts.

In [ ]:
#@title 🎧 Before You Start: Todo3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_36_todo3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 3: Compare Ring-AllReduce vs Tree-AllReduce
#
# Tree-AllReduce:
#   - Number of steps: 2 * log2(N)
#   - Data per step: D/2 (each GPU sends half its data)
#   - Total per-GPU traffic: 2 * log2(N) * D/2 = log2(N) * D
#   - Total time: 2 * log2(N) * (D / (2*B) + L)
#
# Ring-AllReduce:
#   - Number of steps: 2 * (N-1)
#   - Data per step: D/N
#   - Total per-GPU traffic: 2 * (N-1)/N * D ≈ 2D
#   - Total time: 2 * (N-1) * (D / (N*B) + L)
#
# Question: Under what conditions is tree better than ring? (Hint: small messages)

import math

def tree_allreduce_time(N, D_bytes, bandwidth_bps, latency_s):
    """Compute tree-allreduce time."""
    # TODO: implement
    pass

def ring_allreduce_time(N, D_bytes, bandwidth_bps, latency_s):
    """Compute ring-allreduce time."""
    # TODO: implement
    pass

# TODO: Compare the two for different N and D values
# TODO: Plot the comparison


In [ ]:
#@title 🎧 Before You Start: Todo3 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_37_todo3_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("TODO: Implement tree-allreduce and compare with ring-allreduce")
print("Key insight: tree has O(log N) steps (fewer latency hits)")
print("but O(D * log N) bandwidth (worse for large messages)")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_38_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, we have:

1. **Implemented naive allreduce** and seen how it creates a master-GPU bottleneck that grows linearly with the number of GPUs

2. **Built ring-allreduce from scratch** with its two phases: scatter-reduce (accumulate partial sums) and allgather (distribute fully reduced chunks)

3. **Visualized the algorithm** step by step, seeing how each GPU contributes equally to the communication

4. **Analyzed the bandwidth** and showed that ring-allreduce's per-GPU traffic approaches 2D regardless of GPU count (bandwidth-optimal)

5. **Simulated communication time** and demonstrated concrete speedups at realistic scales

6. **Explored gradient bucketing** and how it enables overlapping communication with backward computation

### Key Takeaways

- Naive allreduce has O(N*D) bottleneck traffic -- adding GPUs makes communication SLOWER
- Ring-allreduce has O(D) per-GPU traffic -- adding GPUs barely affects communication time
- Gradient bucketing enables overlapping communication with computation for even better performance
- PyTorch's `DistributedDataParallel` uses ring-allreduce (via NCCL) with bucketing by default

### Next Notebook

In Notebook 2, we will run actual distributed training experiments with different batch sizes and measure how batch size affects convergence, throughput, and the compute-communication trade-off.